# 01. 데이터 탐색

generate_data.py 돌리고 나서 피처가 잘 뽑혔는지 확인하는 용도  
모델 학습 전에 이 노트북 먼저 보는 게 순서

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.data_utils import load_processed_data, print_dataset_stats

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

In [ ]:
agg_df, sequences, seq_labels = load_processed_data('../data/processed')
print_dataset_stats(agg_df, seq_labels)

## 1. 집계 피처 분포 비교
정상 vs 이상 플레이어가 피처별로 얼마나 차이 나는지 확인

In [ ]:
normal = agg_df[agg_df['label'] == 0]
abnormal = agg_df[agg_df['label'] == 1]

key_features = [
    'fold_rate', 'raise_rate', 'allin_rate',
    'strong_fold_rate', 'weak_call_rate',
    'raise_ratio_std', 'avg_allin_hs'
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    ax = axes[i]
    ax.hist(normal[feat].dropna(), bins=30, alpha=0.6, label='정상', color='steelblue')
    ax.hist(abnormal[feat].dropna(), bins=30, alpha=0.6, label='이상', color='tomato')
    ax.set_title(feat)
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('피처별 정상 vs 이상 분포', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 2. 피처 상관관계 히트맵

In [ ]:
feat_cols = [c for c in agg_df.columns if c not in ['player_id', 'label']]
corr = agg_df[feat_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 7})
plt.title('피처 상관관계')
plt.tight_layout()
plt.show()

# 높은 상관관계 피처 쌍 출력 (0.8 이상)
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i):
        if abs(corr.iloc[i, j]) > 0.8:
            high_corr.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))

if high_corr:
    print('높은 상관관계 (|r| > 0.8):')
    for a, b, r in high_corr:
        print(f'  {a} ↔ {b}: {r:.3f}')
else:
    print('다중공선성 우려 피처 없음')

## 3. 시퀀스 피처 시각화
정상/이상 플레이어 액션 시퀀스를 눈으로 비교

In [ ]:
# 정상/이상 각각 샘플 시퀀스 5개씩 뽑아서 hand_strength 흐름 비교
normal_seq = sequences[seq_labels == 0][:5]
abnormal_seq = sequences[seq_labels == 1][:5]

# hand_strength는 인덱스 9번 (encode_action 기준)
HS_IDX = 9

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for seq in normal_seq:
    ax1.plot(seq[:, HS_IDX], alpha=0.6)
ax1.set_title('정상 플레이어 - hand_strength 흐름')
ax1.set_xlabel('액션 순서')
ax1.set_ylabel('hand_strength')
ax1.set_ylim(0, 1)

for seq in abnormal_seq:
    ax2.plot(seq[:, HS_IDX], alpha=0.6, color='tomato')
ax2.set_title('이상 플레이어 - hand_strength 흐름')
ax2.set_xlabel('액션 순서')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()